In [22]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('../data/featured_cloud_billings.csv')

# Drop leaked or unnecessary columns
cols_to_drop = ['unrounded_cost', 'rounded_cost', 'cost_per_quantity', 
                'usage_start_time', 'usage_end_time', 'resource_id', 
                'usage_hour', 'is_weekend']
df = df.drop(columns=cols_to_drop)

In [23]:
# 1. Native Categorical Conversion for XGBoost
categorical_cols = ['service_name', 'usage_unit', 'region_zone']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# 2. Separate target from features
X = df.drop(columns=['total_cost_inr'])
y = df['total_cost_inr']

# 3. Log-transform skewed features
skewed_features = ['usage_quantity', 'usage_duration_hours', 'network_inbound_GB']
for col in skewed_features:
    # Fill missing with 0 temporarily for log transform if needed, though they shouldn't be null
    X[col] = np.log1p(X[col].fillna(0))

# 4. Log-transform the target variable
y_log = np.log1p(y)

In [24]:
# First split: Hold out 15% for the final test evaluation
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_log, test_size=0.15, random_state=42, stratify=X['usage_unit']
)

# Second split: Hold out another 15% from the remainder for Conformal Calibration
X_train, X_calib, y_train, y_calib = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=X_temp['usage_unit']
)

print(f"Training shape: {X_train.shape}")
print(f"Calibration shape: {X_calib.shape}")
print(f"Testing shape: {X_test.shape}")

Training shape: (89788, 10)
Calibration shape: (15845, 10)
Testing shape: (18642, 10)


In [25]:
# Define the quantiles we want to predict (5th, 50th/Median, 95th)
quantiles = [0.05, 0.50, 0.95]

# Initialize the XGBoost Regressor
model = xgb.XGBRegressor(
    objective='reg:quantileerror',
    quantile_alpha=quantiles,
    enable_categorical=True,
    tree_method='hist',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [26]:
# Predict on Calibration Set
# XGBoost returns a 2D array when multiple quantiles are passed: [low, median, high]
preds_calib_log = model.predict(X_calib)
preds_test_log = model.predict(X_test)

# Inverse log transform to get back to INR (Absolute Cost Space)
preds_calib = np.expm1(preds_calib_log)
preds_test = np.expm1(preds_test_log)
y_calib_actual = np.expm1(y_calib).values
y_test_actual = np.expm1(y_test).values

# Extract the specific quantiles for easier handling
calib_low, calib_med, calib_high = preds_calib[:, 0], preds_calib[:, 1], preds_calib[:, 2]
test_low, test_med, test_high = preds_test[:, 0], preds_test[:, 1], preds_test[:, 2]

In [27]:
target_coverage = 0.90
alpha = 1 - target_coverage

# Create a DataFrame for calibration to group by unit easily
calib_df = pd.DataFrame({
    'unit': X_calib['usage_unit'].values,
    'y_true': y_calib_actual,
    'q_low': calib_low,
    'q_high': calib_high
})

# Calculate Conformalized Quantile Regression (CQR) conformity scores
calib_df['score'] = np.maximum(
    calib_df['q_low'] - calib_df['y_true'], 
    calib_df['y_true'] - calib_df['q_high']
)

# Compute the correction factor (Q) per usage_unit
# We find the empirical (1 - alpha) quantile of the scores for each unit
correction_factors = {}
for unit, group in calib_df.groupby('unit'):
    n = len(group)
    if n > 0:
        # Correction index formula for conformal prediction
        q_level = np.ceil((n + 1) * (1 - alpha)) / n
        q_level = min(max(q_level, 0.0), 1.0)
        correction_factors[unit] = group['score'].quantile(q_level)
    else:
        correction_factors[unit] = 0

print("Calibration Correction Factors (INR) per Unit:")
for k, v in correction_factors.items():
    print(f"  {k}: ₹{v:.2f}")

Calibration Correction Factors (INR) per Unit:
  GB: ₹-1378.11
  Hours: ₹-743.38
  Requests: ₹-1302.76


In [29]:
# Apply the specific correction factor to the test set bounds based on the unit
test_df = pd.DataFrame({
    'unit': X_test['usage_unit'].values,
    'y_true': y_test_actual,
    'q_low_raw': test_low,
    'q_med': test_med,
    'q_high_raw': test_high
})

test_df['correction'] = test_df['unit'].astype(object).map(correction_factors).fillna(0.0)

# Calibrated bounds (prevent lower bound from dropping below 0)
test_df['q_low_calib'] = np.maximum(0, test_df['q_low_raw'] - test_df['correction'])
test_df['q_high_calib'] = test_df['q_high_raw'] + test_df['correction']

# Evaluation Metrics
def check_coverage(df, low_col, high_col):
    return ((df['y_true'] >= df[low_col]) & (df['y_true'] <= df[high_col])).mean()

# Calculate Coverages
raw_coverage = check_coverage(test_df, 'q_low_raw', 'q_high_raw')
calib_coverage = check_coverage(test_df, 'q_low_calib', 'q_high_calib')

print(f"Overall Raw Coverage: {raw_coverage * 100:.2f}%")
print(f"Overall Calibrated Coverage: {calib_coverage * 100:.2f}%\n")

print("Calibrated Coverage by Usage Unit:")
for unit, group in test_df.groupby('unit'):
    cov = check_coverage(group, 'q_low_calib', 'q_high_calib')
    print(f"  {unit}: {cov * 100:.2f}%")

# R2 on Median
r2_median = r2_score(np.log1p(test_df['y_true']), np.log1p(test_df['q_med']))
print(f"\nMedian Prediction R² (Log Space): {r2_median:.3f}")

Overall Raw Coverage: 92.75%
Overall Calibrated Coverage: 90.39%

Calibrated Coverage by Usage Unit:
  GB: 90.47%
  Hours: 89.82%
  Requests: 90.68%

Median Prediction R² (Log Space): 0.665


In [30]:
import joblib

# Bundle the model, correction factors, and expected feature schema
export_data = {
    'model': model,
    'correction_factors': correction_factors,
    'features': X.columns.tolist(),
    'categorical_cols': ['service_name', 'usage_unit', 'region_zone']
}

# Save to a file in your project directory
joblib.dump(export_data, '../model/cloud_cost_model.joblib')

print("Model and calibration artifacts exported successfully!")

Model and calibration artifacts exported successfully!
